In [2]:
import pandas as pd
import re
import os

# =================================================================
# 🛠️ 1. 난이도 측정 기준 정의 (WEIGHTS & KEYWORDS)
# =================================================================

# ⭐️ 반복 제한을 적용할 저난이도 키워드를 구분하기 위해 점수별로 그룹화 ⭐️
HIGH_DIFFICULTY_KEYWORDS = {
    # 고난도 기술 (반복 제한 없음)
    '발효': 7, '튀기': 7, '머랭': 7, '중탕': 5, '온도 조절': 5,
    '반죽': 5, '졸이': 4, '굽': 3, '냉동': 3, '숙성': 5, '성형': 6,
    '포뜨': 7, '계량': 4,
    '갈': 4, '데치': 2, 
    # 기본/반복 동작 (점수 낮춤)
    '끓이': 2, '삶': 2, '볶': 1, '다지': 1, '썰': 1, 
}

# ⭐️ 반복 횟수 제한을 적용할 키워드 목록 ⭐️
# (점수가 2점 이하인 기본 동작들)
REPETITIVE_KEYWORDS = ['볶', '끓이', '삶', '데치', '썰', '다지']
MAX_REPETITION_COUNT = 3  # 최대 3회까지만 점수에 반영

LOW_DIFFICULTY_KEYWORDS = {
    '살살': -1, '뭉치지 않게': -1, '주의': -1, '약불로': -1, '쉽게': -1, '곱게': -1, '잘 섞': -1, '간단하게': -1
}

# 가중치 유지
WEIGHTS = {
    'material_score': 0.20,
    'skill_score': 0.35,
    'time_score': 0.15,
    'compression_score': 0.30 
}

CATEGORY_FACTOR = {
    '베이킹': 1.2,
    '김치/발효': 1.15,
    '한식(고난도)': 1.1,
    '중식': 1.05,
    '양식': 1.0,
    '한식(일반)': 1.0,
    '간단식/음료': 0.8
}

# =================================================================
# 💻 2. 난이도 계산 메인 함수
# =================================================================

def calculate_recipe_difficulty(recipe_data):
    """
    레시피 데이터를 분석하여 난이도 점수를 계산하고 분류합니다.
    (반복 동작 점수 기여도 제한 적용)
    """
    
    # 0. 데이터 추출 및 기본값 설정
    ing_count = recipe_data.get('ing_count', 10)
    steps = recipe_data.get('steps', [])
    num_steps = len(steps)
    cook_time = recipe_data.get('cook_time_min', 30) 
    category = recipe_data.get('category', '한식(일반)')
    
    total_skill_points = 0
    high_keyword_count = 0
    low_keyword_score = 0
    
    # 1. 재료 점수 (C_재료) 산출
    if ing_count >= 20:
        material_score = 5
    elif ing_count >= 10:
        material_score = 3
    else:
        material_score = 1
        
    # 2. 조리 시간 점수 (C_시간) 산출
    if cook_time >= 90:
        time_score = 5
    elif cook_time >= 30:
        time_score = 3
    else:
        time_score = 1

    # 3. 기술 점수 (C_기술) 및 키워드 개수 산출 (⭐️ 반복 제한 로직 추가 ⭐️)
    step_content_list = " ".join(steps)
    
    for keyword, score in HIGH_DIFFICULTY_KEYWORDS.items():
        matches = re.findall(rf'{keyword}', step_content_list)
        
        if matches:
            count = len(matches)
            
            # ⭐️ 반복 제한 적용 ⭐️
            if keyword in REPETITIVE_KEYWORDS and count > MAX_REPETITION_COUNT:
                count = MAX_REPETITION_COUNT
            
            total_skill_points += score * count
            high_keyword_count += count

    # 4. 역가중치 (친절한 설명) 점수 산출
    for keyword, score in LOW_DIFFICULTY_KEYWORDS.items():
        low_keyword_score += step_content_list.count(keyword) * score
        
    # 5. 단계 압축도 (S_comp) 산출 (핵심 보정)
    if num_steps > 0:
        compression_score = high_keyword_count / num_steps
    else:
        compression_score = 0
        
    # 6. 최종 점수 계산
    f_cat = CATEGORY_FACTOR.get(category, 1.0)
    
    # 압축도 가중치 (6) 유지
    weighted_score = (
        (WEIGHTS['material_score'] * material_score) +
        (WEIGHTS['skill_score'] * (total_skill_points / 10)) + 
        (WEIGHTS['time_score'] * time_score) +
        (WEIGHTS['compression_score'] * compression_score * 6)
    )
    
    final_score = (weighted_score * f_cat) + low_keyword_score

    # 7. 난이도 분류 (기준 유지: 3.5, 1.5)
    if final_score >= 3.5:
        difficulty_level = "어려움 (숙련자)"
    elif final_score >= 1.5:
        difficulty_level = "보통 (경험자)"
    else: 
        difficulty_level = "쉬움 (초보자)"
        
    return {
        'title': recipe_data.get('title', 'Unknown'),
        'difficulty': difficulty_level,
        'final_score': round(final_score, 2),
        'steps_count': num_steps,
        'skill_points_total': total_skill_points,
        'compression_score': round(compression_score, 2),
    }

# =================================================================
# 🏃 3. 데이터 처리 및 실행 (경로 설정 및 인코딩 포함)
# =================================================================

def run_difficulty_analysis(file_path="C:/Users/alstj/Documents/카카오톡 받은 파일/레시피 스텝.csv", top_n=10):
    
    # --- [사용자 설정 영역] ---
    # ⚠️ 1. CSV 입력 파일 경로 (수정하지 않음)
    
    # ⚠️ 2. CSV 출력 폴더 경로 (수정 가능)
    output_directory = "C:/Users/alstj/Downloads" 
    # --------------------------
    
    try:
        # 출력 디렉토리 생성 (폴더가 없으면 생성)
        os.makedirs(output_directory, exist_ok=True)
        
        # 데이터 로드
        df_raw = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"오류: 파일을 찾을 수 없습니다. 경로를 확인해주세요: {file_path}")
        return

    # 3.1. 데이터 정제: 노이즈 및 코멘트 제거
    comment_patterns = r'^\s*[*\[].*|플리토|외국어 자막|Subtitle|Music provided by'
    df_cleaned = df_raw[
        ~df_raw['레시피'].astype(str).str.contains(comment_patterns, regex=True, na=False)
    ].copy()

    # 3.2. 레시피별로 단계 텍스트 그룹화
    recipe_groups = df_cleaned.groupby(['video_id', 'title'])['레시피'].apply(list).reset_index()
    recipe_groups.columns = ['video_id', 'title', 'steps']

    # 3.3. 알고리즘 적용
    results = []
    for _, row in recipe_groups.iterrows():
        recipe_data = {
            'title': row['title'],
            'steps': row['steps'],
        }
        results.append(calculate_recipe_difficulty(recipe_data))

    # 3.4. 결과 정리 및 저장
    df_results = pd.DataFrame(results)
    df_results_sorted = df_results.sort_values(by='final_score', ascending=False)
    
    output_filename = os.path.join(output_directory, '레시피_난이도_분석_최종_반복제한.csv')
    
    # 엑셀 한글 깨짐 방지 코드 적용: encoding='utf-8-sig' 사용
    df_results_sorted.to_csv(output_filename, index=False, encoding='utf-8-sig')
    
    print(f"\n✅ 분석이 완료되었습니다. 결과는 '{output_filename}' 파일에 저장되었습니다.")
    print(f"\n--- 난이도 분석 결과 (Top {top_n} 레시피) ---")
    print(df_results_sorted.head(top_n).to_markdown(index=False))

# 스크립트 실행
run_difficulty_analysis()


✅ 분석이 완료되었습니다. 결과는 'C:/Users/alstj/Downloads\레시피_난이도_분석_최종_반복제한.csv' 파일에 저장되었습니다.

--- 난이도 분석 결과 (Top 10 레시피) ---
| title                                                                                                        | difficulty      |   final_score |   steps_count |   skill_points_total |   compression_score |
|:-------------------------------------------------------------------------------------------------------------|:----------------|--------------:|--------------:|---------------------:|--------------------:|
| 이러니 맛있을 수박엨ㅋㅋㅋㅋ                                                                                 | 어려움 (숙련자) |          4.2  |             5 |                   18 |                1.4  |
| 떡갈비 버거, 일명 '붹붹버거'입니다! 햄버거는 한식이죠~~?? l 백종원의 쿠킹로그                                | 어려움 (숙련자) |          4.11 |             9 |                   36 |                1    |
| 치즈가 내린다 샤라랄라~♪ 혼자 먹기 너~무 아까운 치즈등갈비                                                   | 어려움 (숙련자) |        